# matrixlayout Binder demo

`matrixlayout` builds LaTeX/TikZ layouts for matrix-centered linear algebra figures and renders them to SVG. The examples below are adapted from the `la_figures` Gaussian-elimination, QR/Gram-Schmidt, eigenproblem, and back-substitution examples, but are written directly against the current `matrixlayout` API.

In [ ]:
from matrixlayout import (
    backsubst_svg,
    decorator_bg,
    decorator_box,
    decorator_color,
    render_eig_svg,
    render_ge_svg,
    render_qr_svg,
    sel_col,
    sel_entry,
    show_svg,
)

RENDER_OPTS = dict(toolchain_name="pdftex_dvisvgm", crop="tight", padding=(2, 2, 2, 2))

## Labels, highlights, and callouts

A reduced matrix can be annotated with row and column labels, highlighted pivot columns, and callouts.

In [ ]:
U = [[1, 2], [0, -2], [0, 0]]

label_svg = render_ge_svg(
    matrices=[U],
    label_rows=[{"grid": (0, 0), "side": "above", "rows": [["$x_1$", "$x_2$"]]}],
    label_cols=[{"grid": (0, 0), "side": "left", "cols": [["$p_1$"], ["$p_2$"], [""]]}],
    decorations=[
        {"grid": (0, 0), "entries": [sel_col(0)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_col(1)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_entry(0, 0), sel_entry(1, 1)], "decorator": decorator_box(color="red")},
        {"grid": (0, 0), "label": "rank 2", "side": "right", "angle": -35, "length": 10},
    ],
    create_medium_nodes=True,
    **RENDER_OPTS,
)
show_svg(label_svg)

## Decorated Jordan form

A single symbolic matrix can be decorated by submatrix ranges. This example highlights and outlines Jordan blocks, then attaches a callout label to the whole matrix.

In [ ]:
import sympy as sym

lam1, lam2 = sym.symbols(r"\lambda_1 \lambda_2")
J1 = sym.Matrix([[lam1, 1], [0, lam1]])
J2 = sym.Matrix([[lam2, 1], [0, lam2]])
J3 = sym.Matrix([[lam2]])
J = sym.diag(J1, J2, J3)

jordan_decorations = [
    {"submatrix": ((0, 1), (0, 1)), "background": "red!10"},
    {"submatrix": ((2, 3), (2, 3)), "background": "yellow!20"},
    {"submatrix": ((4, 4), (4, 4)), "background": "yellow!20"},
    {"submatrix": ("0:1", "0:1"), "outline": True, "padding_pt": 2.0},
    {"submatrix": ("2:3", "2:3"), "outline": True, "padding_pt": 2.0},
    {"submatrix": ("4:4", "4:4"), "outline": True, "padding_pt": 2.0},
    {"label": r"\mathbf{J}", "side": "right", "angle": -35, "length": 8},
]

jordan_svg = render_ge_svg(
    matrices=J,
    decorations=jordan_decorations,
    **RENDER_OPTS,
)
show_svg(jordan_svg)

## Render options

`toolchain_name`, `crop`, and `padding` are forwarded through the rendering boundary. This notebook uses `pdftex_dvisvgm` because it is compact and reliable in Binder.

In [ ]:
compact_svg = render_ge_svg(
    matrices=[[[1, 2], [3, 4]]],
    toolchain_name="pdftex_dvisvgm",
    crop="tight",
    padding=(0, 0, 0, 0),
)
show_svg(compact_svg)

## Matrix stack for a product layout

A matrix stack can show the layout of a computation without computing anything inside `matrixlayout`. This example arranges `A`, `B`, `C`, `AB`, and `ABC` to show the computation `(A B) C`.

In [ ]:
A_stack = [[1, 2, 2], [2, 0, 3]]
B_stack = [[3, 1], [0, 1], [2, 2]]
C_stack = [[2, 2, 1, 1], [-2, -1, 0, -1]]
AB_stack = [[7, 7], [12, 8]]
ABC_stack = [[0, 7, 7, 0], [8, 16, 12, 4]]

stack_matrices = [[None, B_stack, C_stack], [A_stack, AB_stack, ABC_stack]]
stack_annotations = [
    {"grid": (0, 1), "label": "B", "side": "left", "angle_deg": -35, "length_mm": 6},
    {"grid": (0, 2), "label": "C", "side": "above", "angle_deg": -35, "length_mm": 6},
    {"grid": (1, 0), "label": "A", "side": "left", "angle_deg": -35, "length_mm": 6},
    {"grid": (1, 1), "label": "AB", "side": "left", "angle_deg": -35, "length_mm": 12},
    {"grid": (1, 2), "label": "ABC", "side": "above", "angle_deg": -35, "length_mm": 6},
]

stack_svg = render_ge_svg(stack_matrices, annotations=stack_annotations, **RENDER_OPTS)
show_svg(stack_svg)

## Row reduction layout

This is the direct `matrixlayout` input corresponding to `la_figures.ge_tbl_spec(A, gj=false)` for `A = [[1, 2], [1, 2], [3, 4]]`, with `outer_hspace_mm = 8` and right-aligned cells.

In [ ]:
ge_matrices = [
    [None, [[1, 2], [1, 2], [3, 4]]],
    [[[1, 0, 0], [-1, 1, 0], [-3, 0, 1]], [[1, 2], [0, 0], [0, -2]]],
    [[[1, 0, 0], [0, 0, 1], [0, 1, 0]], [[1, 2], [0, -2], [0, 0]]],
]
U = ge_matrices[-1][1]

ge_variable_labels = [{
    "grid": (2, 1),
    "rows": [
        [r"\textcolor{red}{\ensuremath{\Uparrow}}", r"\textcolor{red}{\ensuremath{\Uparrow}}"],
        [r"\textcolor{red}{\ensuremath{x_{1}}}", r"\textcolor{red}{\ensuremath{x_{2}}}"],
    ],
}]
ge_rowechelon_paths = [
    r"\draw[blue,line width=0.5mm] let \p1 = (A0.north west), \p2 = (A0.south east), \p3 = (4-|4), \p4 = (4-|4) in (\x1,\y1) -- (\x4,\y2);",
    r"\draw[blue,line width=0.5mm] let \p1 = (E1.north west), \p2 = (E1.south east), \p3 = (7-|1), \p4 = (7-|1) in (\x1,\y1) -- (\x1,\y2);",
    r"\draw[blue,line width=0.5mm] let \p1 = (A1.north west), \p2 = (A1.south east), \p3 = (5-|4), \p4 = (7-|5) in (\x1,\y1) -- (\x1,\y3) -- ($ (5-|5) + (0.1,0) $) -- ($ (\x4,\y2) + (0.1,0) $);",
    r"\draw[blue,line width=0.5mm] let \p1 = (A2.north west), \p2 = (A2.south east), \p3 = (8-|4), \p4 = (9-|5) in (\x1,\y1) -- (\x1,\y3) -- ($ (8-|5) + (0.1,0) $) -- ($ (9-|5) + (0.1,0) $) -- (\x2,\y4);",
]
ge_codebefore = [
    r"\tikz \node [fill=yellow!40, inner sep = 0pt, fit = (1-4) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 1pt, fit = (1-4) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 1pt, fit = (1-4) (3-4) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 1pt, fit = (4-1) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 1pt, fit = (4-1) (6-1) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 0pt, fit = (4-4) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 0pt, fit = (5-5) ] {} ;",
    r"\tikz \node [fill=gray!20, inner sep = 0pt, fit = (8-3) ] {} ;",
    r"\tikz \node [fill=gray!20, inner sep = 0pt, fit = (9-2) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 0pt, fit = (7-4) ] {} ;",
    r"\tikz \node [fill=yellow!40, inner sep = 0pt, fit = (8-5) ] {} ;",
]

row_reduction_svg = render_ge_svg(
    matrices=ge_matrices,
    variable_labels=ge_variable_labels,
    rowechelon_paths=ge_rowechelon_paths,
    codebefore=ge_codebefore,
    create_cell_nodes=True,
    create_medium_nodes=True,
    outer_hspace_mm=8,
    cell_align="r",
    strict=False,
    **RENDER_OPTS,
)
show_svg(row_reduction_svg)

## GE layout from current structured specs

This example builds the GE layout from a current `GEGridSpec`-style dictionary: matrices, structured entry-background decorations, pivot boxes, row-echelon paths, and basic/free-variable labels.

In [ ]:
current_ge_matrices = [
    [
        None,
        [
            [-1, 2, 1, -1, -1, -6],
            [-1, 1, 3, -4, -4, -6],
            [-1, 4, -4, 6, 6, -5],
        ],
    ],
    [
        [[1, 0, 0], [-1, 1, 0], [-1, 0, 1]],
        [
            [-1, 2, 1, -1, -1, -6],
            [0, -1, 2, -3, -3, 0],
            [0, 2, -5, 7, 7, 1],
        ],
    ],
    [
        [[1, 0, 0], [0, 1, 0], [0, 2, 1]],
        [
            [-1, 2, 1, -1, -1, -6],
            [0, -1, 2, -3, -3, 0],
            [0, 0, -1, 1, 1, 1],
        ],
    ],
]

current_ge_spec = {
    "matrices": current_ge_matrices,
    "decorations": [
        {"grid": (0, 1), "entries": [(0, 0)], "background": "yellow!40"},
        {"grid": (0, 1), "submatrix": ("0:2", "0:0"), "background": "yellow!40", "padding_pt": 1},
        {"grid": (1, 0), "entries": [(0, 0)], "background": "yellow!40", "padding_pt": 1},
        {"grid": (1, 0), "submatrix": ("0:2", "0:0"), "background": "yellow!40", "padding_pt": 1},
        {"grid": (1, 1), "entries": [(0, 0), (1, 1)], "background": "yellow!40"},
        {"grid": (1, 1), "submatrix": ("1:2", "1:1"), "background": "yellow!40", "padding_pt": 1},
        {"grid": (2, 0), "entries": [(1, 1)], "background": "yellow!40", "padding_pt": 1},
        {"grid": (2, 0), "submatrix": ("1:2", "1:1"), "background": "yellow!40", "padding_pt": 1},
        {"grid": (2, 1), "entries": [(0, 0), (1, 1), (2, 2)], "background": "yellow!40"},
    ],
    "pivot_locs": [
        ("(4-4)(4-4)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(5-5)(5-5)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(8-2)(8-2)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(1-4)(1-4)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(7-4)(7-4)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(8-5)(8-5)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(9-6)(9-6)", "draw=red, inner sep=2pt, outer sep=0pt"),
        ("(4-1)(4-1)", "draw=red, inner sep=2pt, outer sep=0pt"),
    ],
    "rowechelon_paths": [
        r"\draw[blue,line width=0.5mm] let \p1 = (A1.north west), \p2 = (A1.south east), \p3 = (5-|4), \p4 = (7-|5) in (\x1,\y1) -- (\x1,\y3) -- ($ (5-|5) + (0.1,0) $) -- ($ (\x4,\y2) + (0.1,0) $);",
        r"\draw[blue,line width=0.5mm] let \p1 = (E2.north west), \p2 = (E2.south east), \p3 = (10-|2), \p4 = (10-|2) in ($ (8-|2) + (0.1,0) $) -- ($ (\x4,\y2) + (0.1,0) $);",
        r"\draw[blue,line width=0.5mm] let \p1 = (A0.north west), \p2 = (A0.south east), \p3 = (4-|4), \p4 = (4-|4) in (\x1,\y1) -- (\x4,\y2);",
        r"\draw[blue,line width=0.5mm] let \p1 = (A2.north west), \p2 = (A2.south east), \p3 = (8-|4), \p4 = (10-|6) in (\x1,\y1) -- (\x1,\y3) -- ($ (8-|5) + (0.1,0) $) -- ($ (9-|5) + (0.1,0) $) -- ($ (9-|6) + (0.1,0) $) -- ($ (\x4,\y2) + (0.1,0) $) -- (\x2,\y2);",
        r"\draw[blue,line width=0.5mm] let \p1 = (E1.north west), \p2 = (E1.south east), \p3 = (7-|1), \p4 = (7-|1) in (\x1,\y1) -- (\x1,\y2);",
    ],
    "variable_labels": [
        {
            "grid": (2, 1),
            "rows": [
                [
                    r"\textcolor{red}{\ensuremath{\Uparrow}}",
                    r"\textcolor{red}{\ensuremath{\Uparrow}}",
                    r"\textcolor{red}{\ensuremath{\Uparrow}}",
                    r"\textcolor{black}{\ensuremath{\uparrow}}",
                    r"\textcolor{black}{\ensuremath{\uparrow}}",
                    "",
                ],
                [
                    r"\textcolor{red}{\ensuremath{x_{1}}}",
                    r"\textcolor{red}{\ensuremath{x_{2}}}",
                    r"\textcolor{red}{\ensuremath{x_{3}}}",
                    r"\textcolor{black}{\ensuremath{x_{4}}}",
                    r"\textcolor{black}{\ensuremath{x_{5}}}",
                    "",
                ],
            ],
        }
    ],
    "create_extra_nodes": True,
    "create_medium_nodes": True,
    "strict": True,
}

current_ge_svg = render_ge_svg(spec=current_ge_spec, **RENDER_OPTS)
show_svg(current_ge_svg)

## Back-substitution cascade

A cascade is a separate output form from matrix grids and tables. This example shows the nested substitution display produced by a REF system with free variables.

In [ ]:
show_backsubstitution_svg = backsubst_svg(
    cascade_txt=[
        r"{\ShortCascade%",
        r"{\ShortCascade%",
        r"{\ShortCascade%",
        r"{$\boxed{ x_2 = \alpha_2,\;x_4 = \alpha_4 }$}%",
        r"{$x_5 = -1$}%",
        r"{$\;\Rightarrow\;\boxed{x_5 = -1}$}%",
        r"}%",
        r"{$x_3 = x_{4} - 1$}%",
        r"{$\;\Rightarrow\;\boxed{x_3 = \alpha_{4} - 1}$}%",
        r"}%",
        r"{$x_1 = - \frac{1}{2} \left( 2 x_{3} - 2 x_{4} + 4 \right)$}%",
        r"{$\;\Rightarrow\;\boxed{x_1 = -1}$}%",
        r"}%",
    ],
    show_system=False,
    show_cascade=True,
    show_solution=False,
    **RENDER_OPTS,
)
show_svg(show_backsubstitution_svg)

## Gram-Schmidt / QR layout

This mirrors `h, mats = nM.gram_schmidt_qr(A; fig_scale=1.1); display(h)` for `A = [-1 -4 -3; 1 0 -3; 1 4 1; -1 0 1]`.

In [ ]:
A_qr = [
    [-1, -4, -3],
    [1, 0, -3],
    [1, 4, 1],
    [-1, 0, 1],
]
W = [
    [-1, -2, -1],
    [1, -2, -1],
    [1, 2, -1],
    [-1, 2, -1],
]
Wt = [
    [-1, 1, 1, -1],
    [-2, -2, 2, 2],
    [-1, -1, -1, -1],
]
WtA = [[4, 8, 0], [0, 16, 16], [0, 0, 4]]
WtW = [[4, 0, 0], [0, 16, 0], [0, 0, 4]]
S = [[(1, 2), 0, 0], [0, (1, 4), 0], [0, 0, (1, 2)]]
Qt = [
    [(-1, 2), (1, 2), (1, 2), (-1, 2)],
    [(-1, 2), (-1, 2), (1, 2), (1, 2)],
    [(-1, 2), (-1, 2), (-1, 2), (-1, 2)],
]
R = [[2, 4, 0], [0, 4, 4], [0, 0, 2]]

qr_svg = render_qr_svg(
    matrices=[[None, None, A_qr, W], [None, Wt, WtA, WtW], [S, Qt, R, None]],
    array_names=True,
    fig_scale=1.1,
    **RENDER_OPTS,
)
show_svg(qr_svg)

## Eigenproblem table

A compact table can show eigenvalues, eigenspace bases, and the eigenbasis matrix for a concrete diagonalizable matrix. Here `A = [[4, 1], [2, 3]]` has eigenvalues `5` and `2`.

In [ ]:
eig_spec = {
    "lambda": [5, 2],
    "ma": [1, 1],
    "evecs": [[[1, 1]], [[-1, 2]]],
    "qvecs": [[[1, 1]], [[-1, 2]]],
}

eig_svg = render_eig_svg(eig_spec, case="S", fig_scale=1.2, **RENDER_OPTS)
show_svg(eig_svg)

## SVD table

This mirrors `svg, spec = nM.show_svd_tbl(A, fig_scale=1.1); display(svg)` for a rank-one rational matrix.

In [ ]:
from fractions import Fraction as F

svd_spec = {
    "sigma": [1, 0],
    "lambda": [1, 0],
    "ma": [2, 2],
    "evecs": [
        [[F(-3, 4), 0, 1, 0], [0, 0, 0, 1]],
        [[0, 1, 0, 0], [F(4, 3), 0, 1, 0]],
    ],
    "qvecs": [
        [[F(-3, 5), 0, F(4, 5), 0], [0, 0, 0, 1]],
        [[0, 1, 0, 0], [F(4, 5), 0, F(3, 5), 0]],
    ],
    "uvecs": [
        [[F(-2, 7), F(-6, 7), F(-3, 7)], [F(3, 7), F(2, 7), F(-6, 7)]],
        [[F(6, 7), F(-3, 7), F(2, 7)]],
    ],
    "sz": (3, 4),
}

svd_svg = render_eig_svg(svd_spec, case="SVD", fig_scale=1.1, **RENDER_OPTS)
show_svg(svd_svg)

## Optional: inspect SVG text

The rendered image is the useful output for most notebook work. Inspect raw SVG only when debugging, saving, or comparing renderer output.

In [ ]:
print(label_svg[:500])

## Troubleshooting

If rendering fails, check that the TeX packages and converters are available. The Binder image checks required TeX packages during `postBuild`, and the renderers keep failure artifacts in a temporary directory when a toolchain fails.

In [ ]:
import shutil

checks = ["latex", "dvisvgm", "kpsewhich"]
{name: shutil.which(name) for name in checks}